# Grain-size ensemble cross-validation

Within each run, spatial groups and buffers prevent same-survey context overlap between training, validation and test for radii 0-3. All methods share the same partitions.

## Channel-ablation setup
Select `RGB_only`, `RGB_slope`, `RGB_elevation`, or `RGB_slope_elevation` with `BEACH_CHANNEL_SETUP` (default: `RGB_slope_elevation`). Training and ensemble evaluation are independent for each setup, retaining identical splits/CV plans, parameter grids, and seeds. Only selected channels are read and normalized; the model input-channel count and saved metadata match the selection. Results are stored in the corresponding setup subfolder. Existing outputs below are retained and may predate this change; rerun to obtain setup-specific results.

**Rerun this notebook: retained historical outputs may describe the previous protocol.**


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.channel_setups import get_channel_setup, setup_result_dir, validate_setup_metadata
from Source.beach_dataset import read_class_name_mapping, read_split_csv
from Source.ensemble import (
    average_probabilities,
    class_weighted_average_probabilities,
    hard_voting_predictions,
    load_model_artifact,
    macro_f1_weights,
    per_class_f1_weights,
    predict_artifact_on_split,
    validate_artifact_compatibility,
    weighted_average_probabilities,
)
from Source.training import classification_metrics, get_device
from Source.cross_validation import evaluate_spatial_ensembles, read_cv_plan

In [ ]:
# Select a setup here, or export BEACH_CHANNEL_SETUP before batch execution.
CHANNEL_SETUP, FEATURE_SUFFIXES = get_channel_setup()
print(f"Channel setup: {CHANNEL_SETUP}; features: {FEATURE_SUFFIXES}", flush=True)

CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
CLASS_MAPPING_PATH = PROJECT_ROOT / "Data" / "Raw" / "grainsize_spec.xlsx"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
OUTPUT_DIR = setup_result_dir(PROJECT_ROOT / "Results" / "gs_ensemble", CHANNEL_SETUP)

CV_DIR = PREPROCESSED_DIR

MODEL_RESULT_DIRS = {
    "convnext_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "gs_convnext_tiny", CHANNEL_SETUP),
    "vit_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "gs_vit_tiny", CHANNEL_SETUP),
    "swin_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "gs_swin_tiny", CHANNEL_SETUP),
    "efficientnetv2_s": setup_result_dir(PROJECT_ROOT / "Results" / "gs_efficientnetv2_s", CHANNEL_SETUP),
}

BATCH_SIZE = 32
NUM_WORKERS = 8

print(f"Project root:   {PROJECT_ROOT}", flush=True)
print(f"Cell data root: {CELL_DATA_ROOT}", flush=True)
print(f"Output dir:     {OUTPUT_DIR}", flush=True)

In [ ]:
samples, validation_splits, manifest = read_cv_plan(CV_DIR)
class_name_by_code = read_class_name_mapping(CLASS_MAPPING_PATH, samples["class"].unique())
print(f"Evaluating four ensemble methods over {manifest['outer_folds']} outer folds", flush=True)

In [ ]:
ensemble_cv = evaluate_spatial_ensembles(
    cv_dir=CV_DIR,
    model_result_dirs=MODEL_RESULT_DIRS,
    output_dir=OUTPUT_DIR,
    feature_suffixes=FEATURE_SUFFIXES,
    channel_setup=CHANNEL_SETUP,
    class_name_by_code=class_name_by_code,
)
display(ensemble_cv["fold_metrics"])

In [ ]:
print("Outer-fold mean and standard deviation", flush=True)
display(ensemble_cv["summary"])

In [ ]:
print("Pooled outer-fold metrics", flush=True)
display(ensemble_cv["pooled_metrics"])

In [ ]:
print("Deployment weights from pooled validation predictions of selected configurations", flush=True)
display(ensemble_cv["deployment_weights"])
display(ensemble_cv["oof_predictions"].head())

In [ ]:
print(f"Fold results: {OUTPUT_DIR / 'cross_validation'}", flush=True)
print(f"Deployment weights: {OUTPUT_DIR / 'ensemble_definition.json'}", flush=True)
print("Prediction times include all four component forward passes plus the voting operation.", flush=True)
print("Component training costs are shared between the four ensemble rules, not four additional fits.", flush=True)

In [ ]:
# Compare every channel setup after each completed ensemble evaluation.
from Source.metrics_reporting import load_channel_ablation_metrics, plot_channel_ablation_metrics

ablation_codes = sorted(int(code) for code in class_name_by_code)
ablation_names = {int(code): name for code, name in class_name_by_code.items()}
channel_ablation_metrics = load_channel_ablation_metrics(
    PROJECT_ROOT / "Results", "gs", ablation_codes, ablation_names,
    expected_cv_manifest=manifest,
)
display(channel_ablation_metrics.style.format(precision=3, na_rep="N/A"))
print("N/A denotes missing or invalid results, not zero performance. See Status and Detail.", flush=True)
plot_channel_ablation_metrics(channel_ablation_metrics, "Grain-size pooled outer-fold channel ablation")
plt.show()
